# 06주차: PyTorch Fashion-MNIST 이미지 분류 기초

## 학습 목표
Fashion-MNIST 흑백 의류 이미지를 텐서로 읽고, 합성곱 신경망으로 열 가지 클래스를 분류합니다. 학습·검증·테스트 데이터의 역할을 구분하고, 로짓·softmax 확률·정확도를 관찰합니다. 학습 전의 무작위 예측과 학습 후 예측을 같은 테스트 이미지에서 비교하며 신경망이 데이터를 통해 바뀌는 과정을 이해합니다.

Colab에서는 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. GPU가 없어도 CPU에서 실행되지만 학습이 오래 걸릴 수 있습니다. Fashion-MNIST는 자동으로 내려받으므로 Google Drive 연결, Drive 마운트, 파일 업로드는 필요하지 않습니다.

## 관찰 질문
- 학습 전 모델의 확률 합은 왜 1이고, 예측은 왜 정답과 자주 다를까요?
- 학습 데이터와 검증 데이터, 테스트 데이터를 분리하는 이유는 무엇일까요?
- 오분류된 두 의류 클래스는 이미지의 어떤 모양 때문에 헷갈릴까요?


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# ---------------------------------------------------------------------------
# 이 노트북의 텐서 차원 표기
#   B = 배치 크기(여기서는 128, 마지막 배치는 더 작을 수 있음)
#   C = 채널 수(Fashion-MNIST는 흑백이라 1)
#   H, W = 이미지의 높이·너비(28 x 28)
#   이미지 텐서는 항상 (B, C, H, W), 라벨 텐서는 항상 (B,) 모양이다.
# ---------------------------------------------------------------------------

def seed_everything(seed=42):
    """난수 생성기를 모두 같은 시드로 고정한다.

    파이썬 random, numpy, PyTorch(CPU/GPU)는 각각 별도의 난수 생성기를 쓴다.
    가중치 초기화·데이터 섞기·증강이 모두 난수를 쓰므로, 넷을 모두 고정해야
    다시 실행했을 때 같은 결과가 나온다.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# GPU가 있으면 GPU를, 없으면 CPU를 쓴다. 이후 모델과 데이터를 모두 이 장치로 보낸다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

def choose_num_workers():
    """실행 환경의 CPU 개수에 맞춰 DataLoader 워커 수를 정한다.

    워커는 배치를 미리 준비해 두는 별도 프로세스다. Colab 런타임마다
    할당되는 vCPU 수가 다르므로(2개인 경우도, 8개인 경우도 있다) 고정값 대신
    실행할 때 세어서 정한다.

    - os.sched_getaffinity(0)은 "이 프로세스가 실제로 쓸 수 있는" 코어를 센다.
      Colab처럼 컨테이너로 코어를 제한하는 환경에서 os.cpu_count()는 호스트
      전체 코어를 세어 과대평가할 수 있다. 리눅스에만 있으므로 없으면 대체한다.
    - 워커를 1개만 쓰면 0개(메인 프로세스가 직접 준비)보다 오히려 느리다.
      병렬성은 없으면서 프로세스 간에 데이터를 넘기는 비용만 붙기 때문이다.
      그래서 코어가 2개 미만이면 아예 0으로 둔다.
    - 8을 넘겨도 이득이 거의 없다. 그 지점이면 이미 GPU 연산이 병목이다.
      코어 수보다 많은 워커를 만들면 오히려 느려지고 PyTorch가 경고도 낸다.
    """
    try:
        cores = len(os.sched_getaffinity(0))     # 리눅스(Colab 포함)
    except AttributeError:
        cores = os.cpu_count() or 1              # macOS, Windows 등
    return 0 if cores < 2 else min(8, cores)

# ToTensor: PIL 이미지(28 x 28, 0~255 정수)를 (1, 28, 28) 모양의 0~1 실수 텐서로 바꾼다.
#           채널 축이 맨 앞에 새로 생긴다는 점에 주의.  (H, W) -> (C, H, W)
transform = transforms.ToTensor()

# 데이터셋의 한 항목은 (이미지 텐서 (1, 28, 28), 라벨 정수) 튜플이다.
# train=True는 학습용 6만 장, train=False는 테스트용 1만 장을 가져온다.
# download=True면 ./data에 없을 때만 내려받고, 이미 있으면 그대로 재사용한다.
full_train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

# 학습용 6만 장을 학습 54,000장과 검증 6,000장으로 쪼갠다.
# generator에 시드를 고정해야 실행할 때마다 같은 이미지가 같은 쪽에 들어간다.
split_generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_train_dataset, [54000, 6000], generator=split_generator)

batch_size = 128
num_workers = choose_num_workers()
print(f"사용 가능한 CPU 코어에 맞춰 num_workers={num_workers}로 설정했습니다.")

# pin_memory: GPU로 옮길 때 빨라지는 메모리 영역을 쓴다(GPU가 있을 때만 의미 있음).
loader_options = {"batch_size": batch_size, "num_workers": num_workers, "pin_memory": torch.cuda.is_available()}

# DataLoader는 낱장 (1, 28, 28)을 batch_size개 쌓아 (B, 1, 28, 28)로 만들어 준다.
# 반복하면 매번 (이미지 (B, 1, 28, 28), 라벨 (B,)) 한 쌍을 돌려준다.
# 학습 데이터만 shuffle=True. 매 에포크 순서를 섞어야 데이터 순서를 외우지 않는다.
# 검증·테스트는 성능을 재기만 하므로 섞을 이유가 없다.
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
print("데이터 수:", len(train_dataset), len(val_dataset), len(test_dataset))


## 데이터 시각화와 모델
`ToTensor`는 28×28 회색조 이미지를 0에서 1 사이의 한 채널 텐서로 바꿉니다. 아래 2×5 표본에서 라벨과 이미지를 함께 확인합니다. `SmallCNN`은 두 개의 합성곱 층과 풀링 층으로 특징을 추출하고, 마지막 분류기가 열 개 클래스의 점수를 만듭니다. 입력과 출력 텐서의 shape도 확인해 보세요.


In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# next(iter(loader))는 배치 하나만 꺼내 본다.
# sample_images: (B, 1, 28, 28)   sample_labels: (B,)
sample_images, sample_labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
# sample_images[:10] -> (10, 1, 28, 28) 을 낱장 (1, 28, 28)씩 꺼내 쓴다.
for image, label, axis in zip(sample_images[:10], sample_labels[:10], axes.flat):
    # imshow는 (H, W) 또는 (H, W, C)만 받는다. image는 (1, 28, 28)이므로
    # squeeze(0)으로 크기가 1인 채널 축을 없앤다.  (1, 28, 28) -> (28, 28)
    axis.imshow(image.squeeze(0), cmap="gray")
    # label은 원소가 하나뿐인 0차원 텐서. .item()으로 파이썬 정수를 꺼낸다.
    axis.set_title(class_names[label.item()])
    axis.axis("off")
plt.tight_layout()
plt.show()
print("입력 텐서 shape:", sample_images.shape)   # (128, 1, 28, 28)

class SmallCNN(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        # features: 이미지에서 "무엇이 보이는지" 특징을 뽑는 부분
        # 아래 주석의 화살표는 그 층을 지난 뒤의 텐서 모양이다.
        self.features = nn.Sequential(
            # Conv2d(입력 채널, 출력 채널=필터 수, 커널 크기, padding)
            # padding=1을 주면 3x3 커널을 써도 H, W가 줄지 않는다. 채널만 1 -> 16으로 바뀐다.
            nn.Conv2d(in_channels, 16, 3, padding=1),   # (B, 1, 28, 28) -> (B, 16, 28, 28)
            nn.ReLU(),                                  # (B, 16, 28, 28) -> (B, 16, 28, 28) 모양 그대로
            nn.MaxPool2d(2),                            # (B, 16, 28, 28) -> (B, 16, 14, 14)  H, W 절반
            nn.Conv2d(16, 32, 3, padding=1),            # (B, 16, 14, 14) -> (B, 32, 14, 14)
            nn.ReLU(),                                  # 모양 그대로
            nn.MaxPool2d(2),                            # (B, 32, 14, 14) -> (B, 32, 7, 7)
            # 입력 H, W가 무엇이든 출력을 4x4로 맞춰준다. 뒤의 Linear 입력 크기를 고정하는 역할.
            nn.AdaptiveAvgPool2d((4, 4)),               # (B, 32, 7, 7) -> (B, 32, 4, 4)
        )
        # classifier: 뽑은 특징을 10개 클래스 점수로 바꾸는 부분
        self.classifier = nn.Sequential(
            # Flatten은 배치 축(0번)만 남기고 나머지를 한 줄로 편다. 32*4*4 = 512
            nn.Flatten(),                               # (B, 32, 4, 4) -> (B, 512)
            nn.Linear(32 * 4 * 4, 64),                  # (B, 512) -> (B, 64)
            nn.ReLU(),                                  # 모양 그대로
            nn.Linear(64, 10),                          # (B, 64) -> (B, 10)  마지막 10 = 클래스 수
        )

    def forward(self, x):
        # x: (B, 1, 28, 28) -> features -> (B, 32, 4, 4) -> classifier -> (B, 10)
        # 반환값은 확률이 아니라 로짓(점수)이다.
        return self.classifier(self.features(x))

# 학습 없이 shape만 확인한다. no_grad()는 기울기 계산을 꺼서 메모리와 시간을 아낀다.
with torch.no_grad():
    shape_logits = SmallCNN(in_channels=1).to(device)(sample_images.to(device))
print("출력 텐서 shape:", shape_logits.shape)   # (128, 10)


## 학습 전 모델 출력 실험
학습되지 않은 모델은 아직 정답을 본 적이 없습니다. 로짓은 클래스별 점수이고, softmax는 이를 확률로 바꾸어 각 이미지의 확률 합을 1로 만듭니다. 전체 테스트셋 초기 정확도는 정확히 10%라고 단정하지 않고 우연 수준에 가까운지 관찰합니다. 같은 `images`를 나중에 학습 후 모델에도 넣어 비교합니다.


In [ ]:
def measure_accuracy(model, loader):
    """loader 전체를 돌며 정확도(맞은 개수 / 전체 개수)를 구한다."""
    model.eval()               # 평가 모드로 전환(Dropout/BatchNorm 동작이 달라진다)
    correct, total = 0, 0
    with torch.no_grad():      # 기울기를 계산하지 않는다. 평가에는 필요 없고 메모리만 쓴다.
        for batch_images, batch_labels in loader:      # (B, 1, 28, 28), (B,)
            logits = model(batch_images.to(device))    # (B, 10)
            # argmax(dim=1): 클래스 축을 따라 최댓값의 위치를 찾는다. (B, 10) -> (B,)
            # (예측 (B,) == 정답 (B,))는 True/False로 된 (B,) 텐서.
            # .sum()은 True의 개수를 담은 0차원 텐서, .item()으로 파이썬 정수를 꺼낸다.
            correct += (logits.argmax(dim=1).cpu() == batch_labels).sum().item()
            total += batch_labels.size(0)              # size(0) = 이 배치의 B
    return correct / total

# 아직 한 번도 학습하지 않은 모델. 가중치는 무작위 초기값 그대로다.
untrained_model = SmallCNN(in_channels=1).to(device)

# 이 images/labels는 뒤에서 학습 후 모델에도 그대로 넣어 비교할 것이므로 변수로 남겨둔다.
images, labels = next(iter(test_loader))       # (B, 1, 28, 28), (B,)

with torch.no_grad():
    initial_logits = untrained_model(images.to(device))          # (B, 10)
    # softmax: 로짓(점수)을 0~1 확률로 바꾸고 합이 1이 되게 만든다. 모양은 그대로 (B, 10).
    # dim=1은 "클래스 축을 따라" 정규화하라는 뜻(이미지 한 장의 10개 점수끼리 합이 1).
    initial_probabilities = torch.softmax(initial_logits, dim=1) # (B, 10)

initial_predictions = initial_probabilities.argmax(dim=1).cpu()  # (B, 10) -> (B,)
initial_accuracy = measure_accuracy(untrained_model, test_loader)

print("로짓 예시:", initial_logits[0].cpu())                      # 첫 이미지의 10개 점수, (10,)
print("확률 합:", initial_probabilities[0].sum().item())          # (10,)를 다 더하면 항상 1.0
print("초기 예측과 정답:", class_names[initial_predictions[0].item()], class_names[labels[0].item()])
# 클래스가 10개이므로 아무렇게나 찍어도 기대값은 10%다. 정확히 10%가 아니어도 괜찮다.
print(f"학습 전 테스트 정확도: {initial_accuracy:.2%}; 우연 수준에 가까운지 해석하세요.")


## 학습·평가 함수와 5에포크 학습
`train_one_epoch`는 한 번의 학습 데이터 순회에서 가중치를 갱신합니다. `evaluate`는 가중치를 바꾸지 않고 손실과 정확도를 계산합니다. `fit`은 매 에포크의 학습·검증 기록을 쌓고, `plot_history`는 변화 추이를 그립니다. CrossEntropyLoss와 Adam 최적화기를 사용합니다.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """학습 데이터를 한 바퀴 돌며 가중치를 갱신한다. (손실, 정확도)를 돌려준다."""
    model.train()                       # 학습 모드
    loss_sum, correct, total = 0.0, 0, 0
    for batch_images, batch_labels in loader:      # (B, 1, 28, 28), (B,)
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)

        # --- 학습 4단계 ---
        optimizer.zero_grad()           # 1. 이전 배치의 기울기를 지운다(안 지우면 누적된다)
        logits = model(batch_images)    # 2. 순전파: (B, 1, 28, 28) -> (B, 10)
        # CrossEntropyLoss는 (B, 10) 로짓과 (B,) 정수 라벨을 받아 0차원 스칼라 손실을 낸다.
        loss = criterion(logits, batch_labels)      # (B, 10), (B,) -> ()
        loss.backward()                 # 3. 역전파: 각 가중치가 손실에 얼마나 기여했는지 계산
        optimizer.step()                # 4. 계산된 기울기 방향으로 가중치를 조금 움직인다

        # loss는 배치 평균이므로, 전체 평균을 내려면 배치 크기를 곱해 더한 뒤 나중에 나눈다.
        loss_sum += loss.item() * batch_labels.size(0)
        correct += (logits.argmax(dim=1) == batch_labels).sum().item()   # (B, 10) -> (B,) 비교
        total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def evaluate(model, loader, criterion, device):
    """가중치를 바꾸지 않고 손실과 정확도만 잰다. train_one_epoch과 달리 backward/step이 없다."""
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_images, batch_labels in loader:      # (B, 1, 28, 28), (B,)
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            logits = model(batch_images)               # (B, 10)
            loss_sum += criterion(logits, batch_labels).item() * batch_labels.size(0)
            correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    """epochs만큼 학습하면서 매 에포크의 학습·검증 기록을 history에 쌓는다.

    history의 각 값은 길이가 epochs인 파이썬 리스트다(텐서가 아니다).
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        # 검증은 학습에 쓰이지 않은 데이터로 잰다. 여기가 오르지 않으면 학습이 헛도는 것.
        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
        for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
            history[key].append(value)
        print(f"Epoch {epoch}/{epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%")
    return history

def plot_history(history, title):
    """왼쪽에 손실, 오른쪽에 정확도를 그린다. 학습 곡선과 검증 곡선을 겹쳐 비교한다."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].legend()
    plt.show()

def count_parameters(model):
    """학습으로 값이 바뀌는 파라미터(가중치·편향)의 총 개수.

    numel()은 그 텐서의 원소 개수다. 예를 들어 Conv2d(1, 16, 3)의 가중치는
    (16, 1, 3, 3) 모양이므로 numel()은 16*1*3*3 = 144가 된다.
    """
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

model = SmallCNN(in_channels=1).to(device)
# CrossEntropyLoss는 내부에서 softmax를 함께 계산한다.
# 그래서 모델은 softmax를 거치지 않은 "로짓" (B, 10)을 그대로 내보내야 한다.
criterion = nn.CrossEntropyLoss()
# Adam: 파라미터마다 학습 속도를 자동으로 조절해 주는 최적화기. lr은 한 번에 움직이는 보폭.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("학습 가능한 파라미터 수:", count_parameters(model))
history = fit(model, train_loader, val_loader, criterion, optimizer, device, epochs=5)
plot_history(history, "SmallCNN")

# 테스트셋은 마지막에 딱 한 번만 쓴다. 이걸 보고 설정을 고치면 테스트셋도 오염된다.
test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
print(f"테스트 손실: {test_loss:.4f}, 테스트 정확도: {test_accuracy:.2f}%")


## 학습 전후 비교와 학생 활동
동일한 테스트 이미지의 초기 예측과 학습 후 예측을 비교하고, 틀린 이미지는 최대 10개만 모아 살펴봅니다. 오분류는 실패 목록이 아니라 다음 실험의 단서입니다. 비슷한 옷의 윤곽, 낮은 해상도, 제한된 모델 용량 중 무엇이 원인인지 추론해 보세요.

### 학생 활동
1. 첫 합성곱 층 뉴런 수 16을 8, 32, 64로 바꾸고 파라미터 수, 시간, 정확도를 비교하세요.
2. 에포크를 1, 5, 10으로 바꾸고 학습·검증 곡선에서 과소적합 또는 과적합의 신호를 설명하세요.
3. 오분류에서 자주 함께 나타나는 두 클래스를 골라 사람이 구분하는 단서를 적어 보세요.

### 7주차 연결 질문
다음 주에는 세 채널 컬러 CIFAR-10에 같은 `SmallCNN` 구조를 적용합니다. `in_channels`만 3으로 바꾸면 어떤 층의 입력이 달라질까요? 데이터 증강과 더 깊은 모델은 오늘의 기본 모델 한계를 어떻게 보완할까요?


In [ ]:
# --- 1) 학습 전과 학습 후를 "같은 이미지"로 비교한다 ---
# images/labels는 학습 전 실험에서 쓴 것과 똑같은 테스트 배치다. (B, 1, 28, 28), (B,)
model.eval()
with torch.no_grad():
    trained_logits = model(images.to(device))          # (B, 10)
trained_predictions = trained_logits.argmax(dim=1).cpu()   # (B, 10) -> (B,)

# initial_predictions와 trained_predictions 모두 (B,)이므로 같은 인덱스가 같은 이미지다.
for index in range(10):
    print(f"{index}: 초기={class_names[initial_predictions[index].item()]}, "
          f"학습 후={class_names[trained_predictions[index].item()]}, "
          f"정답={class_names[labels[index].item()]}")

# --- 2) 학습 후에도 "틀린" 이미지를 최대 10장 모은다 ---
# 아래 그림은 성공 사례가 아니라 오분류(틀린 예측)만 모아 그린 것이다.
# 테스트셋 순서대로 처음 만난 10장이며, "가장 헷갈린 10장"은 아니다.
mistake_images, mistake_predictions, mistake_labels = [], [], []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:                     # (B, 1, 28, 28), (B,)
        batch_predictions = model(batch_images.to(device)).argmax(dim=1).cpu()   # (B,)
        # (batch_predictions != batch_labels)는 (B,) 모양의 True/False 마스크다.
        # 이 마스크로 인덱싱하면 틀린 것만 남는다. 틀린 개수를 K라 하면
        #   batch_images[마스크]      -> (K, 1, 28, 28)
        #   batch_predictions[마스크] -> (K,)
        #   batch_labels[마스크]      -> (K,)
        for image, prediction, label in zip(batch_images[batch_predictions != batch_labels],
                                            batch_predictions[batch_predictions != batch_labels],
                                            batch_labels[batch_predictions != batch_labels]):
            mistake_images.append(image)       # 낱장 (1, 28, 28)
            mistake_predictions.append(prediction)
            mistake_labels.append(label)
            if len(mistake_images) == 10:
                break        # 배치 안쪽 루프 탈출
        if len(mistake_images) == 10:
            break            # 배치 바깥 루프 탈출

# --- 3) 오분류 이미지를 예측/정답과 함께 그린다 ---
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, image, prediction, label in zip(axes.flat, mistake_images, mistake_predictions, mistake_labels):
    axis.imshow(image.squeeze(0), cmap="gray")     # (1, 28, 28) -> (28, 28)
    axis.set_title(f"Pred: {class_names[prediction.item()]}\nActual: {class_names[label.item()]}")
    axis.axis("off")
# 틀린 이미지가 10장이 안 되면 남는 칸은 빈 채로 둔다.
for axis in axes.flat[len(mistake_images):]:
    axis.axis("off")
plt.tight_layout()
plt.show()
